In [1]:
# ============================================================
# STEP 1: Import Required Libraries
# ============================================================

import os
import csv
import time
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, concatenate_datasets
from scipy.stats import pearsonr

from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report
)

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)



In [2]:
from google.colab import drive


# ============================================================
# STEP 2: Basic Settings
# ============================================================

MODEL_NAME = "roberta-base"
TEXT_COL = "text"

EMOTIONS = [
    "anger",
    "fear",
    "joy",
    "sadness",
    "surprise",
    "disgust"
]

INTENSITY_CLASSES = [0, 1, 2, 3]

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16

MAX_LENGTH = 256
LEARNING_RATE = 2e-5

NUM_EPOCHS = 40

EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_THRESHOLD = 0.0

SEED = 42

NUM_LABELS = len(EMOTIONS)


# ============================================================
# STEP 3: Set Random Seed for Reproducibility
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# STEP 4: Set Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# ============================================================
# STEP 5: Mount Google Drive
# ============================================================

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive"
os.makedirs(DRIVE_DIR, exist_ok=True)

LOG_FILE = f"{DRIVE_DIR}/RoBERTa_Intensity_Epoch_Log.csv"
BEST_MODEL_DIR = f"{DRIVE_DIR}/RoBERTa_Intensity_Best_Model"




Mounted at /content/drive


In [5]:
# ============================================================
# STEP 6: Load BRIGHTER English Dataset
# ============================================================

train_original = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_original = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_original = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

print("Original train dataset:")
print(train_original)

print("\nFirst sample:")
print(train_original[0])


# ============================================================
# STEP 7: Combine Dataset and Create 70/20/10 Split
# ============================================================

full_data = concatenate_datasets([
    train_original,
    val_original,
    test_original
])

total_size = len(full_data)

train_size = int(round(0.70 * total_size))
val_size = int(round(0.20 * total_size))
test_size = total_size - train_size - val_size

split_1 = full_data.train_test_split(
    train_size=train_size,
    seed=SEED
)

train_data = split_1["train"]
remaining_data = split_1["test"]

split_2 = remaining_data.train_test_split(
    train_size=val_size,
    seed=SEED
)

val_data = split_2["train"]
test_data = split_2["test"]

print("\nNew dataset split:")
print("Train size:", len(train_data))
print("Validation size:", len(val_data))
print("Test size:", len(test_data))


# ============================================================
# STEP 8: Load RoBERTa Tokenizer
# ============================================================

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


# ============================================================
# STEP 9: Tokenize Text and Create Intensity Labels
# ============================================================

def preprocess(example):
    """
    Tokenizes the text and creates one label vector.

    If any emotion value is missing/None, it is converted to 0.
    """

    encoded = tokenizer(
        example[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH
    )

    labels = []

    for emotion in EMOTIONS:
        value = example[emotion]

        if value is None:
            value = 0

        labels.append(float(value))

    encoded["labels"] = labels

    return encoded


train_tok = train_data.map(preprocess)
val_tok = val_data.map(preprocess)
test_tok = test_data.map(preprocess)

columns_needed = [
    "input_ids",
    "attention_mask",
    "labels"
]

train_tok.set_format(type="torch", columns=columns_needed)
val_tok.set_format(type="torch", columns=columns_needed)
test_tok.set_format(type="torch", columns=columns_needed)




Original train dataset:
Dataset({
    features: ['id', 'text', 'anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise'],
    num_rows: 2763
})

First sample:
{'id': 'eng_train_track_b_00001', 'text': 'Colorado, middle of nowhere.', 'anger': 0, 'disgust': None, 'fear': 1, 'joy': 0, 'sadness': 0, 'surprise': 1}

New dataset split:
Train size: 3950
Validation size: 1129
Test size: 564


Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1129 [00:00<?, ? examples/s]

Map:   0%|          | 0/564 [00:00<?, ? examples/s]

In [6]:
# ============================================================
# STEP 10: Helper Functions for Metrics
# ============================================================

def safe_pearson(x, y):
    """
    Calculates Pearson correlation safely.
    If Pearson cannot be calculated, it returns 0.
    """

    try:
        r, _ = pearsonr(x, y)

        if np.isnan(r):
            return 0.0

        return float(r)

    except Exception:
        return 0.0


def convert_regression_to_classes(values):
    """
    RoBERTa outputs continuous regression values.
    This function converts them into intensity classes.

    Example:
    2.7 becomes 3
    1.3 becomes 1
    -0.5 becomes 0
    4.2 becomes 3
    """

    values = np.asarray(values)
    values = np.rint(values)
    values = np.clip(values, 0, 3)

    return values.astype(int)


# ============================================================
# STEP 11: Compute Metrics
# ============================================================

def compute_metrics(eval_pred):
    """
    Computes evaluation metrics during validation and testing.
    """

    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    pred_classes = convert_regression_to_classes(preds)
    label_classes = convert_regression_to_classes(labels)

    metrics = {}

    pearson_scores = []

    for i, emotion in enumerate(EMOTIONS):
        r = safe_pearson(
            labels[:, i],
            preds[:, i]
        )

        metrics[f"pearson_{emotion}"] = r
        pearson_scores.append(r)

    metrics["pearson_mean"] = float(np.mean(pearson_scores))

    y_true_flat = label_classes.flatten()
    y_pred_flat = pred_classes.flatten()

    metrics["f1_micro"] = f1_score(
        y_true_flat,
        y_pred_flat,
        average="micro",
        labels=INTENSITY_CLASSES,
        zero_division=0
    )

    metrics["f1_macro"] = f1_score(
        y_true_flat,
        y_pred_flat,
        average="macro",
        labels=INTENSITY_CLASSES,
        zero_division=0
    )

    for i, emotion in enumerate(EMOTIONS):
        metrics[f"f1_macro_{emotion}"] = f1_score(
            label_classes[:, i],
            pred_classes[:, i],
            average="macro",
            labels=INTENSITY_CLASSES,
            zero_division=0
        )

    return metrics


# ============================================================
# STEP 12: Create CSV Log File
# ============================================================

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "section",
        "epoch",
        "train_batch_size",
        "eval_batch_size",
        "train_loss",
        "val_loss",
        "val_f1_micro",
        "val_f1_macro",
        "val_pearson_mean",
        "test_loss",
        "test_f1_micro",
        "test_f1_macro",
        "test_pearson_mean",
        "emotion",
        "intensity_class",
        "precision",
        "recall",
        "f1_score",
        "support",
        "best_metric",
        "best_epoch",
        "early_stopped"
    ])

print("CSV log file created at:", LOG_FILE)


# ============================================================
# STEP 13: Lists for Graphs
# ============================================================

epoch_list = []
train_loss_list = []
val_loss_list = []
val_f1_micro_list = []
val_f1_macro_list = []
val_pearson_list = []


# ============================================================
# STEP 14: Callback to Save Epoch-wise Results
# ============================================================

class SaveEpochMetricsCallback(TrainerCallback):
    """
    Saves validation metrics after each epoch into the CSV file.
    """

    def __init__(self, log_file):
        self.log_file = log_file
        self.current_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return

        epoch = metrics.get("epoch", state.epoch)

        if epoch is None:
            return

        epoch = round(float(epoch), 4)

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        epoch_list.append(epoch)
        train_loss_list.append(self.current_train_loss)
        val_loss_list.append(val_loss)
        val_f1_micro_list.append(val_f1_micro)
        val_f1_macro_list.append(val_f1_macro)
        val_pearson_list.append(val_pearson_mean)

        with open(self.log_file, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([
                "validation_epoch",
                epoch,
                "",
                "",
                self.current_train_loss,
                val_loss,
                val_f1_micro,
                val_f1_macro,
                val_pearson_mean,
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                ""
            ])


# ============================================================
# STEP 15: Load RoBERTa Regression Model
# ============================================================

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="regression"
)

model.to(device)


# ============================================================
# STEP 16: Training Arguments
# ============================================================

training_args = TrainingArguments(
    output_dir="/content/roberta_intensity_output",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=1,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=SEED
)


# ============================================================
# STEP 17: Create Trainer with Early Stopping
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tok,
    eval_dataset=val_tok,

    tokenizer=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[
        SaveEpochMetricsCallback(LOG_FILE),
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD
        )
    ]
)


# ============================================================
# STEP 18: Train the Model
# ============================================================

start_time = time.time()

trainer.train()

end_time = time.time()

training_time = end_time - start_time
completed_epochs = trainer.state.epoch

print("\nTraining completed.")
print("Total training time:", round(training_time, 2), "seconds")

if completed_epochs is not None and completed_epochs > 0:
    print("Average time per epoch:", round(training_time / completed_epochs, 2), "seconds")

print("Best validation macro F1:", trainer.state.best_metric)
print("Best model checkpoint:", trainer.state.best_model_checkpoint)


# ============================================================
# STEP 19: Save Training Summary
# ============================================================

early_stopped = False

if completed_epochs is not None:
    early_stopped = completed_epochs < NUM_EPOCHS

best_epoch = ""

if trainer.state.best_model_checkpoint is not None:
    try:
        best_epoch = trainer.state.best_model_checkpoint.split("-")[-1]
    except Exception:
        best_epoch = ""

with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "training_summary",
        completed_epochs,
        TRAIN_BATCH_SIZE,
        EVAL_BATCH_SIZE,
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        trainer.state.best_metric,
        best_epoch,
        early_stopped
    ])

print("Training summary saved to CSV.")


# ============================================================
# STEP 20: Final Test Evaluation
# ============================================================

test_metrics = trainer.evaluate(eval_dataset=test_tok)

print("\nFinal test metrics:")
print(test_metrics)

with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "final_test_summary",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        float(test_metrics.get("eval_loss", 0.0)),
        float(test_metrics.get("eval_f1_micro", 0.0)),
        float(test_metrics.get("eval_f1_macro", 0.0)),
        float(test_metrics.get("eval_pearson_mean", 0.0)),
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        "",
        ""
    ])


# ============================================================
# STEP 21: Get Test Predictions
# ============================================================

test_predictions = trainer.predict(test_tok)

preds = test_predictions.predictions
labels = test_predictions.label_ids

if isinstance(preds, tuple):
    preds = preds[0]

pred_classes = convert_regression_to_classes(preds)
label_classes = convert_regression_to_classes(labels)


# ============================================================
# STEP 22: Save Class-wise Results for Each Emotion
# ============================================================

with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    for i, emotion in enumerate(EMOTIONS):
        y_true = label_classes[:, i]
        y_pred = pred_classes[:, i]

        precision, recall, f1, support = precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=INTENSITY_CLASSES,
            zero_division=0
        )

        for class_index, intensity_class in enumerate(INTENSITY_CLASSES):
            writer.writerow([
                "test_classwise_per_emotion",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                "",
                emotion,
                intensity_class,
                float(precision[class_index]),
                float(recall[class_index]),
                float(f1[class_index]),
                int(support[class_index]),
                "",
                "",
                ""
            ])

print("Class-wise results for each emotion saved to CSV.")


# ============================================================
# STEP 23: Save Overall Class-wise Results Across All Emotions
# ============================================================

y_true_flat = label_classes.flatten()
y_pred_flat = pred_classes.flatten()

precision, recall, f1, support = precision_recall_fscore_support(
    y_true_flat,
    y_pred_flat,
    labels=INTENSITY_CLASSES,
    zero_division=0
)

with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    for class_index, intensity_class in enumerate(INTENSITY_CLASSES):
        writer.writerow([
            "test_classwise_overall",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "",
            "all_emotions",
            intensity_class,
            float(precision[class_index]),
            float(recall[class_index]),
            float(f1[class_index]),
            int(support[class_index]),
            "",
            "",
            ""
        ])

print("Overall class-wise results saved to CSV.")


# ============================================================
# STEP 24: Print Classification Reports
# ============================================================

print("\nOverall Test Classification Report Across All Emotions:")
print(
    classification_report(
        y_true_flat,
        y_pred_flat,
        labels=INTENSITY_CLASSES,
        zero_division=0
    )
)

print("\nEmotion-wise Test Classification Reports:")

for i, emotion in enumerate(EMOTIONS):
    print("\nEmotion:", emotion)
    print(
        classification_report(
            label_classes[:, i],
            pred_classes[:, i],
            labels=INTENSITY_CLASSES,
            zero_division=0
        )
    )


# ============================================================
# STEP 25: Save Best Model and Tokenizer
# ============================================================

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("\nBest model saved at:", BEST_MODEL_DIR)
print("Final CSV log saved at:", LOG_FILE)


# ============================================================
# STEP 26: Plot Training Graphs
# ============================================================

def plot_graph(x, y, title, ylabel):
    plt.figure(figsize=(8, 5))
    plt.plot(x, y, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.show()


plot_graph(
    epoch_list,
    train_loss_list,
    "Training Loss vs Epoch",
    "Training Loss"
)

plot_graph(
    epoch_list,
    val_loss_list,
    "Validation Loss vs Epoch",
    "Validation Loss"
)

plot_graph(
    epoch_list,
    val_f1_micro_list,
    "Validation Micro F1 vs Epoch",
    "Validation Micro F1"
)

plot_graph(
    epoch_list,
    val_f1_macro_list,
    "Validation Macro F1 vs Epoch",
    "Validation Macro F1"
)

plot_graph(
    epoch_list,
    val_pearson_list,
    "Validation Pearson Mean vs Epoch",
    "Validation Pearson Mean"
)

CSV log file created at: /content/drive/MyDrive/RoBERTa_Intensity_Epoch_Log.csv


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'

In [ ]:
def predict_intensities(text: str):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.detach().cpu().numpy()[0]

    discrete = np.clip(np.rint(logits), 0, 3).astype(int)

    return {
        EMOTIONS[i]: {
            "raw": float(logits[i]),
            "intensity_0_3": int(discrete[i])
        }
        for i in range(len(EMOTIONS))
    }

print(predict_intensities("I feel so happy and joyful today!"))